# Building a `pw.x` input with `pw_input.py`

A `pw.x` input file consists of two kinds of blocks:

- **Namelists** — Fortran-style `&NAME ... /` blocks that set scalar parameters (cutoff energies, lattice constants, convergence thresholds, …)
- **Cards** — plain-text keyword blocks that list structured data (atomic species, atomic positions, k-point mesh, …)

`pw_input.py` provides a Python class for each block.  This notebook walks through each one using MgO as an example, then shows how to assemble a complete input ready to be passed to `pw.x`.

In [ ]:
from ase.build import bulk

from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    AtomicSpeciesCard, AtomicPositionsCard, KPointsAutoCard, PWInput,
)

# MgO rocksalt — the same structure used in the convergence notebook
atoms = bulk('MgO', 'rocksalt', a=4.21)
print(atoms)
print('Lattice parameter a =', round(atoms.cell.lengths()[0], 4), 'Å')

---
## Namelists

### `&CONTROL` — run-time settings

`ControlNamelist` sets what kind of calculation to run and where to find and write files.
Only parameters that differ from the QE default need to be given — the rendered output stays minimal.

In [ ]:
control = ControlNamelist(
    calculation='scf',
    prefix='mgo',
    pseudo_dir='./pseudo',
    outdir='./out',
    tprnfor=True,    # print forces
    tstress=True,    # print stress tensor
)
print(control.to_string())

### `&SYSTEM` — crystal structure and basis set

`SystemNamelist` describes the crystal: the Bravais lattice index `ibrav`, the number of atoms and
species, the lattice constants `celldm(1…6)`, and the kinetic-energy cutoff `ecutwfc`.

The `ibrav` value determines which `celldm` entries are required:

| ibrav | Lattice | Parameters needed |
|------:|---------|-------------------|
| 0 | Free (CELL_PARAMETERS card) | none |
| 1 | Simple cubic | `celldm(1)` = a |
| **2** | **Face-centred cubic** | **`celldm(1)` = a** |
| 4 | Hexagonal | `celldm(1)` = a, `celldm(3)` = c/a |
| 14 | Triclinic | `celldm(1…6)` |

MgO rocksalt has FCC symmetry → `ibrav=2`, only `celldm(1) = a` is needed.

`SystemNamelist.from_atoms()` extracts all of this from an ASE `Atoms` object automatically.

In [ ]:
system = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=60)
print(system.to_string())
print()
# celldm_info() shows which lattice parameters ibrav=2 requires
print(system.celldm_info())

### `&ELECTRONS` — SCF solver settings

The default convergence threshold (`conv_thr = 1e-6 Ry`) is adequate for most convergence tests.
An empty `ElectronsNamelist()` renders the block with no explicit parameters, letting QE use its defaults.

In [ ]:
electrons = ElectronsNamelist()
print(electrons.to_string())

# To tighten the threshold:
# ElectronsNamelist(conv_thr=1e-10)

---
## Cards

### `ATOMIC_SPECIES` — element masses and pseudopotentials

Each species needs a label, atomic mass (amu), and the name of its pseudopotential file.
`AtomicSpeciesCard.from_atoms()` looks up masses from ASE and maps symbols to filenames via a dict.

In [ ]:
PSEUDOS = {
    'Mg': 'Mg.upf',
    'O':  'O.upf',
}

species = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS)
print(species.to_string())

### `ATOMIC_POSITIONS` — where the atoms are

Positions can be given in several units.  `crystal` (fractional coordinates) is the most portable choice:
it does not change when the lattice constant is varied, which matters when we sweep `ecutwfc`.

In [ ]:
positions = AtomicPositionsCard.from_atoms(atoms, units='crystal')
print(positions.to_string())

### `K_POINTS {automatic}` — Brillouin zone sampling

A Monkhorst–Pack mesh is specified by three integers nk₁×nk₂×nk₃, one per reciprocal lattice direction.
How many independent integers are needed depends on the crystal symmetry.

MgO rocksalt has `ibrav=2` (FCC). The three FCC primitive lattice vectors have equal lengths and equal
angles between them — the reciprocal lattice has the same cubic symmetry. All three mesh directions are
equivalent, so a single integer `nk` fully specifies the nk×nk×nk grid.

`KPointsAutoCard` enforces this: for `ibrav=2` it accepts only `nk`.  Passing `nk1`, `nk2`, `nk3`
raises a `TypeError` immediately.

> **Note** — By default the grid is Γ-centred (`sk1=sk2=sk3=0`).  For metals you may want
> `sk1=sk2=sk3=1` to shift it off Γ.

In [ ]:
kpoints = KPointsAutoCard(2, nk=4)   # 4×4×4 mesh for ibrav=2 (FCC)
print(kpoints.to_string())
print()
print(repr(kpoints))
print()
# Passing nk1/nk2/nk3 for a cubic lattice raises a clear error:
try:
    KPointsAutoCard(2, nk1=4, nk2=4, nk3=4)
except TypeError as e:
    print('TypeError:', e)

---
## Assembling a complete input

`PWInput` collects all the namelist and card objects and renders a complete `pw.x` input string
in the correct order.

In [ ]:
inp = PWInput(
    control=control,
    system=system,
    electrons=electrons,
    atomic_species=species,
    atomic_positions=positions,
    k_points=kpoints,
)
print(inp.to_string())

---
## Running a single calculation with `QERunner`

`QERunner.run_one(tag, inp, run_dir)` runs `pw.x` on one input, writes `<tag>.in` / `<tag>.out`
to `run_dir`, and returns a result dict with all quantities pre-parsed.
On a second call with the same tag it detects the existing output and skips the run (cache).

| key | type | meaning |
|-----|------|---------|
| `tag` | `str` | run name |
| `energy_ry` | `float` | total energy (Ry) |
| `wall_s` | `float` | wall time in s; `nan` if loaded from cache |
| `nk_irr` | `int \| None` | irreducible k-points |
| `forces_ev_ang` | `ndarray (nat, 3) \| None` | atomic forces (eV/Å); requires `tprnfor=True` |
| `stress_kbar` | `ndarray (3, 3) \| None` | full stress tensor (kbar); requires `tstress=True` |
| `pressure_kbar` | `float \| None` | hydrostatic pressure (kbar) |
| `scf_corrections_ev_ang` | `ndarray (nat, 3) \| None` | per-atom SCF force errors (eV/Å); requires `verbosity='medium'` |

Any quantity absent from the `pw.x` output (flag not set) comes back as `None`.

In [ ]:
from pathlib import Path
import glob, shutil, os

from convergence_runner import QERunner, RY_TO_EV

os.environ['OMP_NUM_THREADS'] = '1'

_pw_candidates = sorted(glob.glob('/home/pietro/repositories/q-e/build_test_gcc/bin/pw.x'))
PW_CMD = [_pw_candidates[0]] if _pw_candidates else (
    [shutil.which('pw.x')] if shutil.which('pw.x') else None
)
if PW_CMD is None:
    raise RuntimeError('pw.x not found — adjust the glob pattern above.')
print('pw.x:', PW_CMD[0])

RUN_DIR = Path('out')
RUN_DIR.mkdir(exist_ok=True)

# Add verbosity='medium' to also collect SCF force corrections
control_demo = ControlNamelist(
    calculation='scf',
    prefix='mgo_demo',
    pseudo_dir='./pseudo',
    outdir='./out',
    tprnfor=True,
    tstress=True,
    verbosity='medium',
)
inp_demo = PWInput(
    control=control_demo,
    system=SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=60),
    electrons=ElectronsNamelist(),
    atomic_species=AtomicSpeciesCard.from_atoms(atoms, PSEUDOS),
    atomic_positions=AtomicPositionsCard.from_atoms(atoms, units='crystal'),
    k_points=KPointsAutoCard(2, nk=4),
)

runner = QERunner(PW_CMD)
result = runner.run_one('mgo_demo', inp_demo, RUN_DIR)

### Inspecting the result

`result` is a plain dict.  Extract the quantities you need directly by key:

In [ ]:
import numpy as np

nat = len(atoms)

wall_str = 'cached' if np.isnan(result['wall_s']) else f"{result['wall_s']:.1f} s"
print(f"tag          : {result['tag']}")
print(f"energy       : {result['energy_ry']:.6f} Ry  =  {result['energy_ry'] * RY_TO_EV / nat:.4f} eV/atom")
print(f"wall time    : {wall_str}")
print(f"irred. k-pts : {result['nk_irr']}")
print()

forces = result['forces_ev_ang']
print('Forces (eV/Å):')
if forces is not None:
    for i, (fx, fy, fz) in enumerate(forces, 1):
        print(f'  atom {i:2d}   {fx:+.6f}  {fy:+.6f}  {fz:+.6f}')
    print(f'  max|F| = {np.max(np.abs(forces)):.3e} eV/Å')
else:
    print('  not available (set tprnfor=True in &CONTROL)')
print()

print(f"Pressure     : {result['pressure_kbar']:.4f} kbar"
      if result['pressure_kbar'] is not None
      else "Pressure     : not available (set tstress=True in &CONTROL)")
print()

stress = result['stress_kbar']
print('Stress tensor (kbar):')
if stress is not None:
    for label, row in zip('xyz', stress):
        print(f'  {label}   ' + '  '.join(f'{v:+9.4f}' for v in row))
else:
    print('  not available (set tstress=True in &CONTROL)')
print()

corr = result['scf_corrections_ev_ang']
print('SCF force corrections (eV/Å):')
if corr is not None:
    for i, (cx, cy, cz) in enumerate(corr, 1):
        print(f'  atom {i:2d}   {cx:+.6f}  {cy:+.6f}  {cz:+.6f}')
    print(f'  max|dF| = {np.max(np.abs(corr)):.3e} eV/Å')
else:
    print('  not available (set verbosity="medium" in &CONTROL)')

---
## Exercise

In the convergence notebook you will complete `build_mgo_input(atoms, ecutwfc, nk, prefix)` — a function
that wraps exactly these steps and returns a `PWInput`.

### Part A

Look at the input printed above.  Identify which lines come from which namelist or card object.

### Part B

Rebuild the input with `ecutwfc=30`, `nk=6`, and `prefix='mgo_test'`.  Print the result.  What changed
and what stayed the same?

<details>
<summary><b>Solution</b></summary>

```python
inp2 = PWInput(
    control   = ControlNamelist(calculation='scf', prefix='mgo_test',
                                pseudo_dir='./pseudo', outdir='./out',
                                tprnfor=True, tstress=True),
    system    = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=30),
    electrons = ElectronsNamelist(),
    atomic_species   = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS),
    atomic_positions = AtomicPositionsCard.from_atoms(atoms, units='crystal'),
    k_points  = KPointsAutoCard(2, nk=6),
)
print(inp2.to_string())
```

The `prefix` and `ecutwfc` lines in `&CONTROL` and `&SYSTEM` change; the k-mesh line changes;
species, positions, and most namelist parameters stay the same.

</details>

### Part C

The pseudopotentials used in the convergence notebook are norm-conserving.  For norm-conserving
pseudopotentials, QE sets `ecutrho = 4 × ecutwfc` automatically — you do not need to specify it.
For ultrasoft pseudopotentials the default is not sufficient: `ecutrho` typically needs to be
8–12 × ecutwfc.  How would the `&SYSTEM` block change in that case, and what additional convergence
test would you need to run?

### Part D

The `&ELECTRONS` block above is empty (all defaults).  The default SCF convergence threshold is
`conv_thr = 1e-6 Ry`.  The convergence notebook targets 5 meV/atom ≈ 3.7 × 10⁻⁴ Ry per cell.
Is the default `conv_thr` appropriate?  How many orders of magnitude tighter than the energy
target should it be, and why?